In [ ]:
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing import image
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Conv2D, MaxPooling2D
from tensorflow.python.keras.layers import Activation, Dropout, Flatten, Dense
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

ModuleNotFoundError: No module named 'keras'

# Сплитим фотки для обучения и валидации модели

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

true_files = sorted((DATA_DIR / "true").glob("*.JPG"))
false_files = sorted((DATA_DIR / "false").glob("*.JPG"))

X_full = np.array(true_files + false_files, dtype=object)
y_full = np.array([1]*len(true_files) + [0]*len(false_files), dtype=int)

# Сплитим фото 80/10/10
X_train, X_tmp, y_train, y_tmp = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

print("Total:", len(X_full), "pos%:", y_full.mean())
print("Train:", len(X_train), "pos%:", y_train.mean())
print("Val:  ", len(X_val), "pos%:", y_val.mean())
print("Test: ", len(X_test), "pos%:", y_test.mean())

# Preprocessor

In [ ]:
img_width, img_height = 150, 150
batch_size = 10
input_shape = (img_width, img_height, 3)

def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (img_height, img_width))
    img = tf.cast(img, tf.float32) / 255.0
    return img

def make_dataset(X, y, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(str), y))
    if training:
        ds = ds.shuffle(len(X), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, lbl: (preprocess_image(p), tf.cast(lbl, tf.float32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val,   training=False)
test_ds  = make_dataset(X_test,  y_test,  training=False)

# Создаем сверточную нейронную сеть

In [ ]:
# параметры нейронной сети
def build_model(input_shape):
    model = Sequential()

    # Слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
    # Слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(32, (3, 3), input_shape=input_shape))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Слой свертки, размер ядра 3х3, количество карт признаков - 32 шт., функция активации ReLU
    # Слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(32, (3, 3)))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Слой свертки, размер ядра 3х3, количество карт признаков - 64 шт., функция активации ReLU.
    # Слой подвыборки, выбор максимального значения из квадрата 2х2
    model.add(Conv2D(64, (3, 3)))
    model.add(Activation("relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    # Слой преобразования из двумерного в одномерное представление
    # Полносвязный слой, 64 нейрона, функция активации ReLU
    # Слой Dropout
    # Выходной слой, 1 нейрон, функция активации sigmoid
    model.add(Flatten())
    model.add(Dense(64))
    model.add(Activation("relu"))
    model.add(Dropout(0.5))
    model.add(Dense(1))
    model.add(Activation("sigmoid"))

    return model

model = build_model(input_shape)

# компиляция нейронной сети
model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

model.summary()

# Baseline: Stratified K-Fold (optimizator fix, CV: epochs 10/20)

In [ ]:
# 1) dev набор (без test)
X_dev = np.concatenate([X_train, X_val])
y_dev = np.concatenate([y_train, y_val])

# 2) настройки CV
EPOCH_CANDIDATES = [10, 20]
K = 5
seed = 42
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=seed)

def cv_epochs_only(X, y, epochs_list):
    results = {}

    for epochs in epochs_list:
        fold_acc = []

        for tr_idx, va_idx in skf.split(X, y):
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_va, y_va = X[va_idx], y[va_idx]

            tr_ds = make_dataset(X_tr, y_tr, training=True)
            va_ds = make_dataset(X_va, y_va, training=False)

            model = build_model(input_shape)
            model.compile(
                loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                metrics=["accuracy"],
            )

            model.fit(tr_ds, epochs=epochs, validation_data=va_ds, verbose=0)
            _, acc = model.evaluate(va_ds, verbose=0)
            fold_acc.append(acc)

        results[epochs] = (float(np.mean(fold_acc)), float(np.std(fold_acc)))

    return results

res = cv_epochs_only(X_dev, y_dev, EPOCH_CANDIDATES)
print(res)  # {10: (mean,std), 20: (mean,std)}

best_epochs = max(res, key=lambda e: res[e][0])
print("BEST epochs:", best_epochs)

# Baseline: Stratified K-Fold (epochs fix, cv: optimizator Adam/RMSProp/SGD)

In [2]:
# dev набор (без test)
X_dev = np.concatenate([X_train, X_val])
y_dev = np.concatenate([y_train, y_val])

K = 5
seed = 42
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=seed)

# фиксируем epochs, найденные на прошлом шаге
EPOCHS = best_epochs  # например 10 или 20

optimizers = {
    "adam":    tf.keras.optimizers.Adam(learning_rate=1e-3),
    "rmsprop": tf.keras.optimizers.RMSprop(learning_rate=1e-3),
    "sgd":     tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.0),
}

def cv_optimizers(X, y, optimizers_dict, epochs):
    results = {}

    for name, opt in optimizers_dict.items():
        fold_acc = []

        for tr_idx, va_idx in skf.split(X, y):
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_va, y_va = X[va_idx], y[va_idx]

            tr_ds = make_dataset(X_tr, y_tr, training=True)
            va_ds = make_dataset(X_va, y_va, training=False)

            model = build_model(input_shape)
            model.compile(
                loss="binary_crossentropy",
                optimizer=opt,
                metrics=["accuracy"],
            )

            model.fit(tr_ds, epochs=epochs, validation_data=va_ds, verbose=0)
            _, acc = model.evaluate(va_ds, verbose=0)
            fold_acc.append(acc)

        results[name] = (float(np.mean(fold_acc)), float(np.std(fold_acc)))

    return results

res_opt = cv_optimizers(X_dev, y_dev, optimizers, EPOCHS)
print(res_opt)  # {"adam":(mean,std), ...}

best_opt_name = max(res_opt, key=lambda k: res_opt[k][0])
print("BEST optimizer:", best_opt_name)

NameError: name 'np' is not defined

# Final model (epoch = , optimizator = )

In [ ]:
# финальные параметры (интерпретация)
# best_epochs: число эпох, которое в среднем даёт лучшую валидацию на K-Fold (без test)
# best_opt_name: оптимизатор, который в среднем даёт лучшую валидацию при best_epochs

FINAL_EPOCHS = best_epochs
FINAL_OPTIMIZER_NAME = best_opt_name

# 1) финальное обучение на dev (train+val), тест — только один раз в конце
X_dev = np.concatenate([X_train, X_val])
y_dev = np.concatenate([y_train, y_val])

train_final_ds = make_dataset(X_dev, y_dev, training=True)
test_ds = make_dataset(X_test, y_test, training=False)

# собрать и скомпилировать модель
final_model = build_model(input_shape)

if FINAL_OPTIMIZER_NAME == "adam":
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
elif FINAL_OPTIMIZER_NAME == "rmsprop":
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=1e-3)
else:  # "sgd"
    optimizer = tf.keras.optimizers.SGD(learning_rate=1e-2, momentum=0.0)

final_model.compile(
    loss="binary_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

# 2) обучение
history = final_model.fit(
    train_final_ds,
    epochs=FINAL_EPOCHS,
    verbose=1
)

# 3) финальная оценка на test
loss, acc = final_model.evaluate(test_ds, verbose=0)
print(f"FINAL params: epochs={FINAL_EPOCHS}, optimizer={FINAL_OPTIMIZER_NAME}")
print(f"FINAL test_acc: {acc:.4f} | test_loss: {loss:.4f}")

# Интерпретация результатов